# S5.4 — Response Caching (Redis)

Interactive verification that the Redis cache client works correctly:
- Deterministic SHA256 key generation
- Cache store/retrieve round-trip with RAGResponse
- Graceful degradation on Redis failure
- Cache invalidation and stats

In [1]:
import sys
sys.path.insert(0, "../..")
print("\u2713 sys.path configured")

✓ sys.path configured


## 1. Cache Key Generation

Verify deterministic SHA256 keys from query parameters.

In [2]:
from unittest.mock import AsyncMock
from src.services.cache.client import CacheClient

mock_redis = AsyncMock()
client = CacheClient(redis_client=mock_redis, ttl_seconds=86400)

# Same params -> same key
key1 = client._generate_cache_key("What are transformers?", "gemini-2.0-flash", 5, None)
key2 = client._generate_cache_key("What are transformers?", "gemini-2.0-flash", 5, None)
assert key1 == key2, "Keys should be identical"
print(f"Key: {key1}")

# Normalization: whitespace + case
key3 = client._generate_cache_key("  What Are Transformers?  ", "gemini-2.0-flash", 5, None)
assert key1 == key3, "Normalized keys should match"

# Category order doesn't matter
key4 = client._generate_cache_key("q", "m", 5, ["cs.AI", "cs.CL"])
key5 = client._generate_cache_key("q", "m", 5, ["cs.CL", "cs.AI"])
assert key4 == key5, "Sorted categories should match"

# Prefix and length
assert key1.startswith("rag:response:"), "Key should have prefix"
assert len(key1) == len("rag:response:") + 64, "SHA256 = 64 hex chars"

print("\n\u2713 Cache key generation: deterministic, normalized, fixed-length")

Key: rag:response:146e379a2cc8cb0e0f8e027b9363f1bb1a0eb1d27e6edb8253746ce80d3598bd

✓ Cache key generation: deterministic, normalized, fixed-length


## 2. Store & Retrieve Round-Trip

Verify RAGResponse serialization/deserialization through the cache.

In [3]:
import asyncio
from src.services.rag.models import RAGResponse, SourceReference, RetrievalMetadata, LLMMetadata

# Create a sample response
response = RAGResponse(
    answer="Transformers use self-attention [1].",
    sources=[
        SourceReference(
            index=1,
            arxiv_id="1706.03762",
            title="Attention Is All You Need",
            authors=["Vaswani", "Shazeer"],
            arxiv_url="https://arxiv.org/abs/1706.03762",
            chunk_text="The dominant sequence transduction models...",
            score=0.95,
        ),
    ],
    query="What are transformers?",
    retrieval_metadata=RetrievalMetadata(stages_executed=["hybrid_search", "rerank"]),
    llm_metadata=LLMMetadata(provider="gemini", model="gemini-2.0-flash"),
)

# Mock Redis to capture stored data and return it on get
stored = {}

async def mock_set(key, value, **kwargs):
    stored[key] = value

async def mock_get(key):
    return stored.get(key)

mock_redis = AsyncMock()
mock_redis.set = AsyncMock(side_effect=mock_set)
mock_redis.get = AsyncMock(side_effect=mock_get)

client = CacheClient(redis_client=mock_redis, ttl_seconds=3600)

# Store
await client.store_response("What are transformers?", "gemini-2.0-flash", 5, response, None)
print(f"Stored {len(stored)} key(s)")

# Retrieve
cached = await client.find_cached_response("What are transformers?", "gemini-2.0-flash", 5, None)
assert cached is not None, "Should find cached response"
assert cached.answer == response.answer
assert cached.sources[0].arxiv_id == "1706.03762"
assert cached.sources[0].title == "Attention Is All You Need"
print(f"Retrieved: {cached.answer[:50]}...")
print(f"Source: {cached.sources[0].title}")

# Miss
miss = await client.find_cached_response("different query", "gemini-2.0-flash", 5, None)
assert miss is None, "Should return None on miss"

print("\n\u2713 Store/retrieve round-trip: serialization intact, miss returns None")

Stored 1 key(s)
Retrieved: Transformers use self-attention [1]....
Source: Attention Is All You Need

✓ Store/retrieve round-trip: serialization intact, miss returns None


## 3. Graceful Degradation

Verify that Redis failures never propagate exceptions.

In [4]:
# Redis that always fails
failing_redis = AsyncMock()
failing_redis.get = AsyncMock(side_effect=ConnectionError("Redis down"))
failing_redis.set = AsyncMock(side_effect=ConnectionError("Redis down"))
failing_redis.delete = AsyncMock(side_effect=ConnectionError("Redis down"))
failing_redis.dbsize = AsyncMock(side_effect=ConnectionError("Redis down"))

failing_client = CacheClient(redis_client=failing_redis, ttl_seconds=3600)

# None of these should raise
result = await failing_client.find_cached_response("q", "m", 5, None)
assert result is None, "Should return None on error"

await failing_client.store_response("q", "m", 5, response, None)  # no exception

count = await failing_client.invalidate("q", "m", 5, None)
assert count == 0, "Should return 0 on error"

stats = await failing_client.get_stats()
assert stats["keys_count"] == 0, "Should return 0 on error"

print("\u2713 Graceful degradation: all operations return safely on Redis failure")

Cache lookup failed (graceful skip)
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/cache/client.py", line 55, in find_cached_response
    cached = await self._redis.get(key)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
ConnectionError: Redis down
Cache store failed (graceful skip)
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/cache/client.py", line 76, in store_response
    await self._redis.set(key, data, ex=self._ttl)
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
ConnectionError: Redis down
Cache invalidation failed (graceful skip)
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/cache/client.py", line 91, in 

✓ Graceful degradation: all operations return safely on Redis failure


## 4. Factory Functions

Verify factory creation with mocked Redis.

In [5]:
from unittest.mock import patch
from src.services.cache.factory import make_redis_client, make_cache_client

# Success case
mock_instance = AsyncMock()
mock_instance.ping.return_value = True

with patch("src.services.cache.factory.Redis", return_value=mock_instance):
    redis_client = await make_redis_client()
    assert redis_client is not None
    print("make_redis_client() success: returned client")

# Failure case
mock_instance.ping.side_effect = ConnectionError("Refused")
with patch("src.services.cache.factory.Redis", return_value=mock_instance):
    redis_client = await make_redis_client()
    assert redis_client is None
    print("make_redis_client() failure: returned None (graceful)")

print("\n\u2713 Factory functions work correctly")

Redis unavailable (caching disabled)
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/cache/factory.py", line 33, in make_redis_client
    await client.ping()
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
ConnectionError: Refused


make_redis_client() success: returned client
make_redis_client() failure: returned None (graceful)

✓ Factory functions work correctly


## 5. Dependency Injection

Verify `CacheDep` is available in the DI container.

In [6]:
from src.dependency import CacheDep, get_cache_client, set_cache_client

# Initially None
assert get_cache_client() is None, "Should be None before startup"

# Set a mock client
mock_client = CacheClient(redis_client=AsyncMock(), ttl_seconds=3600)
set_cache_client(mock_client)
assert get_cache_client() is mock_client, "Should return the set client"

# Reset
set_cache_client(None)
assert get_cache_client() is None, "Should be None after reset"

print("\u2713 CacheDep available in dependency injection")
print("\n=== S5.4 Response Caching — All Checks Passed ===")

✓ CacheDep available in dependency injection

=== S5.4 Response Caching — All Checks Passed ===
